# 00 — Database Schema Setup
## SentinelMesh IoT Security Platform

**Purpose:** Bootstrap the entire PostgreSQL database schema for SentinelMesh. **Run this notebook once, before all others.** It creates every table and index used across all downstream notebooks.

**Database:** `sentinelmesh_db` (PostgreSQL 15) | **Connection:** `.env` credentials + SQLAlchemy engine

**Schema Overview:**

| Section | Table(s) Created | Purpose |
|---|---|---|
| Staging Table | `stg_harmonized` | Single ingestion layer — all ETL notebooks write here |
| Dimension Table | `dim_device` | Device registry, one row per unique device |
| Fact Tables | `fact_device_risk_daily`, `fact_attack_timeline` | Daily risk aggregations and attack trend analysis |
| ML Logging Tables | `ml_predictions`, `ml_cross_dataset_eval`, `ml_fewshot_sweep` | Model output storage for all pipeline phases |
| Indexes | Multiple | Query performance optimisation on key join/filter columns |
| Schema Verification | — | Confirms all tables exist and are accessible before pipeline runs |

**Pipeline Execution Order:**
1. `00_schema_setup.ipynb` ← **YOU ARE HERE**
2. `01_data_exploration.ipynb`
3. `02_model_training.ipynb`
4. `03_ton_iot_etl.ipynb`
5. `04_prediction_writeback.ipynb`
6. `05_cross_dataset_eval.ipynb`

---

In [1]:
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()

DB_HOST = os.environ.get("DB_HOST")
DB_PORT = os.environ.get("DB_PORT")
DB_NAME = os.environ.get("DB_NAME")
DB_USER = os.environ.get("DB_USER")
DB_PASSWORD = os.environ.get("DB_PASSWORD")

if not all([DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD]):
    raise RuntimeError("Missing one or more DB_* variables in your .env file.")

DB_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(DB_URL)

with engine.connect() as con:
    db_name = con.execute(text("SELECT current_database()")).scalar()
    db_version = con.execute(text("SELECT version()")).scalar()

print(f"✅ Connected to: {db_name}")
print(f"   PostgreSQL: {db_version.split(',')[0]}")

✅ Connected to: sentinelmesh_db
   PostgreSQL: PostgreSQL 15.18 (Homebrew) on aarch64-apple-darwin25.4.0


## Staging Table
`stg_harmonized` is the single ingestion layer for all datasets (CICIoT2023 and TON_IoT).
All ETL notebooks write here. Schema is harmonised across both dataset formats.

In [2]:
with engine.begin() as con:
    con.execute(text("""
        CREATE TABLE IF NOT EXISTS stg_harmonized (
            id                  SERIAL PRIMARY KEY,
            dataset_source      VARCHAR(32)  NOT NULL,
            source_file         VARCHAR(255),
            sample_seed         INTEGER,
            phase               INTEGER,

            -- Core network flow fields (shared across datasets)
            protocol            VARCHAR(32),
            duration            DOUBLE PRECISION,
            src_bytes           BIGINT,
            dst_bytes           BIGINT,
            total_bytes         BIGINT,
            src_pkts            BIGINT,
            dst_pkts            BIGINT,
            total_pkts          BIGINT,

            -- CICIoT2023-specific flow statistics
            flow_rate_byts_s    DOUBLE PRECISION,
            flow_rate_pkts_s    DOUBLE PRECISION,
            syn_flag            INTEGER,
            ack_flag            INTEGER,
            fin_flag            INTEGER,
            rst_flag            INTEGER,
            cic_iat_mean        DOUBLE PRECISION,
            cic_iat_std         DOUBLE PRECISION,
            cic_flow_pkts_s     DOUBLE PRECISION,
            cic_active_mean     DOUBLE PRECISION,
            cic_idle_mean       DOUBLE PRECISION,

            -- TON_IoT-specific application layer fields
            ton_conn_state      VARCHAR(64),
            ton_dns_query       TEXT,
            ton_http_method     VARCHAR(32),

            -- Labels
            label_raw           VARCHAR(128),
            unified_label       VARCHAR(128),
            attack_family       VARCHAR(128),
            is_attack           BOOLEAN,
            ingest_timestamp    TIMESTAMP DEFAULT NOW()
        )
    """))
    print("✅ stg_harmonized created")

✅ stg_harmonized created


## Dimension Table
`dim_device` is the device registry. One row per unique device seen across all ingested data.

In [3]:
with engine.begin() as con:
    con.execute(text("""
        CREATE TABLE IF NOT EXISTS dim_device (
            device_key          SERIAL PRIMARY KEY,
            source_device_id    VARCHAR(128) UNIQUE,
            device_type         VARCHAR(64),
            vendor              VARCHAR(64),
            environment         VARCHAR(64),
            first_seen_at       TIMESTAMP
        )
    """))
    print("✅ dim_device created")

✅ dim_device created


## Fact Tables
`fact_device_risk_daily` aggregates per-device, per-day risk scores driven by ML predictions.
`fact_attack_timeline` aggregates attack events per day for temporal trend analysis.

In [4]:
with engine.begin() as con:
    con.execute(text("""
        CREATE TABLE IF NOT EXISTS fact_device_risk_daily (
            id                      SERIAL PRIMARY KEY,
            device_id               VARCHAR(64)  NOT NULL,
            risk_date               DATE         NOT NULL,
            dataset_source          VARCHAR(32)  NOT NULL,

            -- Flow counts
            total_flows             INTEGER,
            attack_flows            INTEGER,
            benign_flows            INTEGER,
            attack_ratio            DOUBLE PRECISION,

            -- Attack profile
            top_attack_family       VARCHAR(128),
            unique_attack_types     INTEGER,

            -- Volume metrics
            total_bytes             BIGINT,
            total_pkts              BIGINT,
            avg_duration            DOUBLE PRECISION,
            dominant_protocol       VARCHAR(32),

            -- Risk scoring
            -- Formula: 0.60 * attack_ratio + 0.25 * (unique_attack_types/10).clip(0,1)
            --         + 0.15 * attack_ratio — scaled 0–100
            -- NOTE: weights are provisional; see review notes for calibration improvement
            risk_score              DOUBLE PRECISION,
            risk_tier               VARCHAR(16),   -- Low / Medium / High / Critical

            ingest_timestamp        TIMESTAMP DEFAULT NOW()
        )
    """))
    print("✅ fact_device_risk_daily created")

✅ fact_device_risk_daily created


In [5]:
with engine.begin() as con:
    con.execute(text("""
        CREATE TABLE IF NOT EXISTS fact_attack_timeline (
            id                  SERIAL PRIMARY KEY,
            event_date          DATE         NOT NULL,
            dataset_source      VARCHAR(32)  NOT NULL,
            attack_family       VARCHAR(128),
            unified_label       VARCHAR(128),
            protocol            VARCHAR(32),
            flow_count          INTEGER,
            total_bytes         BIGINT,
            total_pkts          BIGINT,
            avg_duration        DOUBLE PRECISION,
            ingest_timestamp    TIMESTAMP DEFAULT NOW()
        )
    """))
    print("✅ fact_attack_timeline created")

✅ fact_attack_timeline created


## ML Logging Tables
`ml_prediction_log` — one row per model-per-flow inference result.
`ml_model_registry` — one row per model training run summary.
`ml_cross_dataset_eval` — one row per cross-dataset transferability result (Notebook 05).

In [6]:
with engine.begin() as con:
    con.execute(text("""
        CREATE TABLE IF NOT EXISTS ml_prediction_log (
            prediction_id       SERIAL PRIMARY KEY,
            stg_id              BIGINT,
            dataset_source      VARCHAR(32),
            model_name          VARCHAR(64),
            model_version       VARCHAR(32),
            predicted_label     INTEGER,

            -- predicted_prob: raw classification probability P(attack)
            -- anomaly_score: currently identical to predicted_prob
            --   TODO: replace with unsupervised scorer (e.g. Isolation Forest)
            --   for true anomaly semantics
            predicted_prob      NUMERIC(8,4),
            anomaly_score       NUMERIC(8,4),

            ground_truth        INTEGER,
            attack_family       VARCHAR(128),
            unified_label       VARCHAR(128),
            correct             BOOLEAN,
            scored_at           TIMESTAMP DEFAULT NOW()
        )
    """))
    print("✅ ml_prediction_log created")

✅ ml_prediction_log created


In [7]:
with engine.begin() as con:
    con.execute(text("""
        CREATE TABLE IF NOT EXISTS ml_model_registry (
            registry_id         SERIAL PRIMARY KEY,
            model_name          VARCHAR(64),
            model_version       VARCHAR(32),
            scored_at           DATE,
            total_rows          INTEGER,
            weighted_f1         NUMERIC(8,4),
            weighted_prec       NUMERIC(8,4),
            weighted_rec        NUMERIC(8,4),
            cic_f1              NUMERIC(8,4),
            ton_f1              NUMERIC(8,4),
            is_champion         BOOLEAN,
            registered_at       TIMESTAMP DEFAULT NOW()
        )
    """))
    print("✅ ml_model_registry created")

✅ ml_model_registry created


In [8]:
with engine.begin() as con:
    con.execute(text("""
        CREATE TABLE IF NOT EXISTS ml_cross_dataset_eval (
            eval_id             SERIAL PRIMARY KEY,
            model_name          VARCHAR(64),
            model_version       VARCHAR(32),
            train_dataset       VARCHAR(32),
            eval_dataset        VARCHAR(32),
            cic_f1              NUMERIC(8,4),
            ton_f1              NUMERIC(8,4),
            delta_abs           NUMERIC(8,4),
            delta_rel_pct       NUMERIC(8,2),
            precision_score     NUMERIC(8,4),
            recall_score        NUMERIC(8,4),
            champion            BOOLEAN,
            evaluated_at        TIMESTAMP DEFAULT NOW()
        )
    """))
    print("✅ ml_cross_dataset_eval created")

✅ ml_cross_dataset_eval created


In [9]:
with engine.begin() as con:
    con.execute(text("""
        CREATE TABLE IF NOT EXISTS ml_domain_adaptation (
            id                SERIAL PRIMARY KEY,
            experiment        VARCHAR(50),
            train_data        VARCHAR(100),
            model             VARCHAR(50),
            ton_weighted_f1   NUMERIC(6,4),
            ton_benign_f1     NUMERIC(6,4),
            f1_delta_vs_zero  NUMERIC(6,4),
            evaluated_at      TIMESTAMP DEFAULT NOW()
        )
    """))
print("✅ ml_domain_adaptation created")

✅ ml_domain_adaptation created


## Indexes
Performance indexes for all high-traffic query patterns across Metabase dashboards,
Streamlit views, and notebook queries.

In [10]:
indexes = {
    # stg_harmonized — filtered constantly by dataset and label
    "idx_stg_dataset_source":       "CREATE INDEX IF NOT EXISTS idx_stg_dataset_source       ON stg_harmonized(dataset_source)",
    "idx_stg_unified_label":        "CREATE INDEX IF NOT EXISTS idx_stg_unified_label        ON stg_harmonized(unified_label)",
    "idx_stg_is_attack":            "CREATE INDEX IF NOT EXISTS idx_stg_is_attack            ON stg_harmonized(is_attack)",
    "idx_stg_ingest_timestamp":     "CREATE INDEX IF NOT EXISTS idx_stg_ingest_timestamp     ON stg_harmonized(ingest_timestamp)",

    # dim_device — lookups by source_device_id
    "idx_dim_device_source_id":     "CREATE INDEX IF NOT EXISTS idx_dim_device_source_id     ON dim_device(source_device_id)",

    # fact_device_risk_daily — dashboard time series and device drilldowns
    "idx_risk_device_date":         "CREATE INDEX IF NOT EXISTS idx_risk_device_date         ON fact_device_risk_daily(device_id, risk_date)",
    "idx_risk_dataset_source":      "CREATE INDEX IF NOT EXISTS idx_risk_dataset_source      ON fact_device_risk_daily(dataset_source)",
    "idx_risk_tier":                "CREATE INDEX IF NOT EXISTS idx_risk_tier                ON fact_device_risk_daily(risk_tier)",

    # fact_attack_timeline — time-series attack trend charts
    "idx_timeline_date_source":     "CREATE INDEX IF NOT EXISTS idx_timeline_date_source     ON fact_attack_timeline(event_date, dataset_source)",
    "idx_timeline_attack_family":   "CREATE INDEX IF NOT EXISTS idx_timeline_attack_family   ON fact_attack_timeline(attack_family)",

    # ml_prediction_log — model comparison and per-dataset F1 queries
    "idx_pred_model_source":        "CREATE INDEX IF NOT EXISTS idx_pred_model_source        ON ml_prediction_log(model_name, dataset_source)",
    "idx_pred_scored_at":           "CREATE INDEX IF NOT EXISTS idx_pred_scored_at           ON ml_prediction_log(scored_at)",
    "idx_pred_stg_id":              "CREATE INDEX IF NOT EXISTS idx_pred_stg_id              ON ml_prediction_log(stg_id)",

    # ml_model_registry — champion lookups
    "idx_registry_model_name":      "CREATE INDEX IF NOT EXISTS idx_registry_model_name      ON ml_model_registry(model_name)",
    "idx_registry_is_champion":     "CREATE INDEX IF NOT EXISTS idx_registry_is_champion     ON ml_model_registry(is_champion)",

    # ml_cross_dataset_eval — transferability result lookups
    "idx_cross_eval_model_name":    "CREATE INDEX IF NOT EXISTS idx_cross_eval_model_name    ON ml_cross_dataset_eval(model_name)",
}

with engine.begin() as con:
    for name, stmt in indexes.items():
        con.execute(text(stmt))
        print(f"  ✅ {name}")

print(f"\n✅ All {len(indexes)} indexes created")

  ✅ idx_stg_dataset_source
  ✅ idx_stg_unified_label
  ✅ idx_stg_is_attack
  ✅ idx_stg_ingest_timestamp
  ✅ idx_dim_device_source_id
  ✅ idx_risk_device_date
  ✅ idx_risk_dataset_source
  ✅ idx_risk_tier
  ✅ idx_timeline_date_source
  ✅ idx_timeline_attack_family
  ✅ idx_pred_model_source
  ✅ idx_pred_scored_at
  ✅ idx_pred_stg_id
  ✅ idx_registry_model_name
  ✅ idx_registry_is_champion
  ✅ idx_cross_eval_model_name

✅ All 16 indexes created


## Schema Verification
Final audit — confirms all tables exist and reports their disk size.

In [11]:
import pandas as pd

audit_sql = text("""
SELECT
    t.table_name,
    pg_size_pretty(pg_total_relation_size(quote_ident(t.table_name))) AS table_size,
    (SELECT COUNT(*) FROM information_schema.columns c
     WHERE c.table_name = t.table_name AND c.table_schema = 'public') AS column_count
FROM information_schema.tables t
WHERE t.table_schema = 'public'
  AND t.table_name IN (
      'stg_harmonized',
      'dim_device',
      'fact_device_risk_daily',
      'fact_attack_timeline',
      'ml_prediction_log',
      'ml_model_registry',
      'ml_cross_dataset_eval'
  )
ORDER BY t.table_name;
""")

index_audit_sql = text("""
SELECT
    indexname,
    tablename
FROM pg_indexes
WHERE schemaname = 'public'
  AND indexname LIKE 'idx_%'
ORDER BY tablename, indexname;
""")

with engine.connect() as con:
    df_tables  = pd.DataFrame(con.execute(audit_sql).fetchall(),
                               columns=['table_name', 'table_size', 'column_count'])
    df_indexes = pd.DataFrame(con.execute(index_audit_sql).fetchall(),
                               columns=['indexname', 'tablename'])

print("=" * 55)
print("  SentinelMesh — Database Schema Audit")
print("=" * 55)
print(f"\n  Tables ({len(df_tables)}):")
print(df_tables.to_string(index=False))
print(f"\n  Indexes ({len(df_indexes)}):")
print(df_indexes.to_string(index=False))
print("\n✅ Schema bootstrap complete — safe to run pipeline notebooks")

  SentinelMesh — Database Schema Audit

  Tables (7):
            table_name table_size  column_count
            dim_device      96 kB             6
  fact_attack_timeline     392 kB            11
fact_device_risk_daily    1832 kB            17
 ml_cross_dataset_eval      40 kB            13
     ml_model_registry      56 kB            12
     ml_prediction_log     244 MB            14
        stg_harmonized     222 MB            32

  Indexes (33):
                 indexname              tablename
  idx_dim_device_source_id             dim_device
idx_timeline_attack_family   fact_attack_timeline
  idx_timeline_date_source   fact_attack_timeline
   idx_risk_dataset_source fact_device_risk_daily
      idx_risk_device_date fact_device_risk_daily
             idx_risk_tier fact_device_risk_daily
          idx_pred_correct    fact_ml_predictions
            idx_pred_model    fact_ml_predictions
           idx_fact_attack      fact_network_flow
           idx_fact_family      fact_network_

In [12]:
with engine.connect() as con:
    result = con.execute(text("""
        SELECT table_name FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name
    """)).fetchall()
    for r in result:
        print(r[0])

dim_attack_type
dim_dataset_source
dim_device
dim_protocol
fact_attack_timeline
fact_device_risk_daily
fact_ml_predictions
fact_network_events
fact_network_flow
ml_cross_dataset_eval
ml_domain_adaptation
ml_model_registry
ml_prediction_log
pg_stat_statements
pg_stat_statements_info
raw_cic_iot_events
raw_ton_iot_events
stg_cic_raw
stg_harmonized
stg_ton_raw
vw_attack_distribution
vw_attack_trend_by_family
vw_model_performance
vw_protocol_risk
vw_severity_summary
vw_top_attacks
